# YouTube to WebDAV
This script allows for a YouTube video to be downloaded and then automatically uplaoded via WebDAV

Usecase is useful for running Syncplay or similar software in situations where running yt-dlp may not be viable

This approach can still fail in cases where JS solver is required

# What this does?
- Downloads vidoe via yt-dlp
- Downloads all official subtitles (non-auto-generated)
- Mux into single MKV file
- Upload via WebDav

Optionally if you are using Nextcloud you can also automatically generate a share

# Usage
Fill in all the fields below. Then `Run all`

In [ ]:
# Install the dependencies needed
!pip install yt-dlp webdavclient3

In [ ]:
from webdav3.client import Client
webdav_url = ""  #@param {type:"string"}
username = ""  #@param {type:"string"}
password = ""  #@param {type:"string"}
options = {
    'webdav_hostname': webdav_url,
    'webdav_login':    username,
    'webdav_password': password
}
client = Client(options)
try:
  client.list()
  print("Success! Connection successful")
except:
  print("Login failed. Please check that your login and WebDAV url are correct")

In [ ]:
youtube_url = "" #@param{type:"string"}

!yt-dlp -f 'bv*+ba/best' \
  --merge-output-format mkv \
  --embed-subs \
  --write-subs \
  --sub-langs all \
  "$youtube_url"

In [ ]:
# Uploads the file to the path specified
download_path = "" #@param{type:"string"}

import glob

mkv_files = glob.glob("/content/*.mkv")
sub_files = glob.glob("/content/*.vtt") + glob.glob("/content/*.srt")
if not mkv_files:
    raise FileNotFoundError("MKV file not found! Did the download fail?")
original_mkv_path = mkv_files[0]
remote_filename = f"{download_path}/" + original_mkv_path.split("/")[-1]
client.upload_sync(remote_path=remote_filename, local_path=original_mkv_path)

print(f"Done! Your file has been uploaded to {remote_filename}")

# OPTIONAL NEXTCLOUD AUTO SHARE
- This uses Nextcloud API to create a public share URL automatically. Completes the workflow

In [ ]:
import requests
nextcloud_url = "" #@param {type:"string"}
api_url = f"{nextcloud_url}/ocs/v2.php/apps/files_sharing/api/v1/shares"
payload = {
    "path": remote_filename,
    "shareType": 3,        # 3 = public link
    "permissions": 31      # optional, full permissions (read/write/share)
}

headers = {
    "OCS-APIRequest": "true"
}
response = requests.post(api_url, auth=(username, password), headers=headers, data=payload)
if response.status_code == 200:
    import xml.etree.ElementTree as ET
    root = ET.fromstring(response.content)
    ns = {"ocs": "http://open-collaboration-services.org/ns"}
    url_elem = root.find(".//url")
    if url_elem is not None:
        share_link = url_elem.text
        print("Share link:", share_link+"/download")
    else:
        print("Could not find share URL in response")
else:
    print("Error:", response.status_code, response.text)

In [ ]:
# Cleanup/Delete old files
import os
mkv_files = glob.glob("/content/*.mkv")
sub_files = glob.glob("/content/*.vtt") + glob.glob("/content/*.srt")
for f in mkv_files + sub_files:
    os.remove(f)
    print(f"Deleted local file: {f}")